In [ ]:
# import dependencies
%matplotlib inline
import os
import sys
import spatioev as sv
import matplotlib
import scanpy as sc
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from spatioev.config import ClusteringConfig

In [ ]:
# set a working directory
wdir = ('/Users/shihongwu/SpatioEv')
os.chdir(wdir)

In [ ]:
adata = sv.load_h5ad("data/exp_1.h5ad")
adata

In [ ]:
# 1) Read annotations CSV (saved from adata_final.obs.to_csv, so index is first column)
ann = pd.read_csv("results/svm_phenotyping_results.csv", index_col=0)

# 2) Optional: keep only columns you want
ann = ann[["annotated_clusters_update3", "svm_prediction"]]

# 3) Join into adata.obs by cell ID (index)
adata.obs = adata.obs.join(ann, how="left")

In [ ]:
sv.tl.estimate_density_adaptive_dbscan_params(adata,
                                                label_key="annotated_clusters_update3",
                                                label_value="tumour")

In [ ]:
adata = sv.tl.cluster_spatial_niches(adata,
                                 label_key="annotated_clusters_update3",
                                 label_value="tumour",
                                 eps=45,
                                 min_samples=5,)

In [ ]:
sv.pl.plot_spatial_category(
    adata,
    feature="tumour_component",
    image_id=None,          # or one image id like "exp"
    image_key="imageid",
    x_key="X_centroid",
    y_key="Y_centroid",
    point_size=6,
    alpha=0.85,
    palette="tab20",
    show_legend=False,      # important for long component lists
    figsize=(10, 10),
)
plt.close('all')

In [ ]:
# Build tumour niche boundaries
#
# Parameter tuning notes for method="density_mask":
# mask_resolution: smaller -> more detailed but can fragment the geometry
#                   larger  -> smoother / coarser boundary
# mask_sigma:      smaller -> preserves fissures and sharp local detail
#                   larger  -> makes the nest more solid and merges nearby gaps
# mask_threshold:  smaller -> larger / more inclusive nest region
#                   larger  -> tighter / more conservative boundary
# mask_closing_size: smaller -> keeps fine structure
#                    larger  -> fills small gaps and connects nearby fragments
#
# Practical guide:
# if the boundary is too fragmented: increase mask_sigma or mask_closing_size
# if the boundary is too smooth: decrease mask_sigma or mask_closing_size
# if shrinking fails later: the geometry is still too thin / fragmented locally
#
boundary_df = sv.tl.build_niche_boundaries(
    adata,
    component_key="tumour_component",
    min_cluster_size=20,
    method="density_mask",
    mask_resolution=6.0,
    mask_sigma=2.0,
    mask_threshold=0.08,
    mask_closing_size=9,
)

buffered_boundary_df = sv.tl.buffer_niche_boundaries(
    boundary_df,
    component_key="tumour_component",
    expand_by=30 / 0.325,
    shrink_by=30 / 0.325,
)

assignments_df = sv.tl.assign_cells_to_niche_regions(
    adata,
    buffered_boundary_df,
    component_key="tumour_component",
    image_key="imageid",
    x_key="X_centroid",
    y_key="Y_centroid",
    region_key="tumour_region",
    mode="distance_to_edge",
    boundary_width=30 / 0.325,
)

In [ ]:
sv.pl.plot_niche_boundaries(
    adata,
    buffered_boundary_df,
    image_id="exp",
    image_key="imageid",
    x_key="X_centroid",
    y_key="Y_centroid",
    point_size=4,
    point_color="lightgray",
    point_alpha=0.6,
    boundary_color="black",
    expanded_color="red",
    shrunk_color="blue",
    figsize=(10, 10),
)
plt.close('all')

In [ ]:
adata = sv.add_niche_regions_to_obs(
    adata,
    assignments_df,
    region_key="tumour_region",
    component_key="tumour_component",
)

In [ ]:
sv.pl.plot_spatial_category(
    adata,
    feature="tumour_region",
    image_id="exp",
    image_key="imageid",
    x_key="X_centroid",
    y_key="Y_centroid",
    point_size=6,
    alpha=0.9,
    palette="tab20",
    show_legend=True,
    figsize=(10, 10),
)
plt.close('all')